# DriveDE - 3D Roundabout Render (Blender on Colab)

Renders the procedural low-poly roundabout explainer (German rules: yield to ring traffic,
enter without signaling, signal right to exit) as 480 frames / 16 s at 720x1280.

**How to run:** Runtime -> Change runtime type -> **T4 GPU**, then Runtime -> **Run all**.
Takes ~25-35 min total (Blender download ~3 min, render ~20-30 min on the T4).
The last cell downloads `roundabout-3d.mp4`. Send it to Claude Code for compositing.


In [ ]:
# @title Step 1: Download Blender + the scene script
import os
if not os.path.exists('/content/blender'):
    !wget -q https://download.blender.org/release/Blender4.2/blender-4.2.3-linux-x64.tar.xz -O /content/blender.tar.xz
    !tar -xf /content/blender.tar.xz -C /content
    !mv /content/blender-4.2.3-linux-x64 /content/blender
    !apt-get -qq install -y libxi6 libxrender1 libxkbcommon0 libsm6 > /dev/null
!wget -q -O /content/roundabout.py https://raw.githubusercontent.com/abhijit5721/DriveDE/staging/content/blender/roundabout.py
!/content/blender/blender --version | head -1
print('setup done')


In [ ]:
# @title Step 2: Render all 480 frames on the GPU (~20-30 min)
!mkdir -p /content/frames
!/content/blender/blender --background --python /content/roundabout.py -- --animate --out /content/frames --samples 48 2>&1 | grep -E "Fra:1 |Fra:100 |Fra:200 |Fra:300 |Fra:400 |Fra:480 |RENDER DONE" | tail -10
import glob
n = len(glob.glob('/content/frames/*.png'))
print(f'{n} frames rendered')
assert n >= 480, 'render incomplete - check the log above'


In [ ]:
# @title Step 3: Encode + download roundabout-3d.mp4
!ffmpeg -y -loglevel error -framerate 30 -i /content/frames/f_%04d.png -c:v libx264 -pix_fmt yuv420p -crf 16 /content/roundabout-3d.mp4
import os
print(os.path.getsize('/content/roundabout-3d.mp4'), 'bytes')
from google.colab import files
files.download('/content/roundabout-3d.mp4')
